# Final Project Notebook

Academic notebook for predicting the future price direction of financial assets with machine learning and a trading backtest.

This notebook starts with: setup, data collection, data loading, and data processing. The modeling sections come after the data is validated, so every later result is tied to a clear and reproducible dataset.


Predicting BUY / HOLD / SELL direction for a future horizon.
It combines stock OHLCV prices with broader market and macro / alternative signals, then compares two model families, LSTM and XGBoost.

The full workflow is:

1. Set up the Python project environment.
2. Load the project configuration.
3. Collect historical stock, benchmark, macro / alternative, and earnings data.
4. Process the raw data into a supervised learning table.
5. Validate date coverage, merged external features, labels, and missing values.
6. Train and compare LSTM and XGBoost models.
7. Evaluate statistical metrics and financial backtest metrics after costs.
8. Run inference for a sample ticker and list demo-ready output files.


## 1. Setup

This cell finds the project root, loads `configs/config.yaml`, and derives notebook defaults from the same configuration used by training, inference, and the UI.

The key point is that `HORIZON` is not hard-coded here. It uses `default_prediction_horizon` from the config, so if the config changes from 21 trading days to another trained horizon, the notebook follows automatically. `SAMPLE_TICKER` also comes from the first ticker in the configured training universe.

Training uses the device setting from config. `device: "auto"` chooses CUDA on NVIDIA, MPS/Metal on Apple Silicon Mac, and CPU otherwise.

Processed datasets are cached under `data/cache/` after the first build. Keep `use_dataset_cache: true` for faster reruns. Set `force_rebuild_dataset_cache: true` in config when tickers, dates, macro sources, thresholds, or feature logic changed.

Keep `RUN_TRAINING = True` for a full fresh run. Set `REBUILD_DATASET = True` when you want the notebook data-preview cell to force a new data download and rebuild only the processed dataset before training.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
import yaml

# Find the project root so the notebook works from Colab, Jupyter, or the notebooks folder.
ROOT = Path.cwd()
if not (ROOT / 'configs' / 'config.yaml').exists():
    ROOT = ROOT.parent
os.chdir(ROOT)

# Load the same project configuration used by train.py, predict.py, and the UI.
config_path = ROOT / 'configs' / 'config.yaml'
with open(config_path, 'r', encoding='utf-8') as f:
    config = yaml.safe_load(f)

available_horizons = [int(h) for h in config.get('prediction_horizons', [config.get('prediction_horizon', 21)])]
HORIZON = int(config.get('default_prediction_horizon', available_horizons[0]))
if HORIZON not in available_horizons:
    HORIZON = available_horizons[0]

SAMPLE_TICKER = str(config['tickers'][0])
RUN_TRAINING = True
INSTALL_REQUIREMENTS = True
REBUILD_DATASET = False

print(f'Project root: {ROOT}')
print(f'Config file: {config_path}')
print(f'Configured horizons: {available_horizons}')
print(f'Selected notebook horizon: {HORIZON} trading days')
print(f'Sample ticker from config: {SAMPLE_TICKER}')
print(f'Rebuild dataset first: {REBUILD_DATASET}')


In [ ]:
def run_command(args, check=True):
    print('\n$ ' + ' '.join(str(a) for a in args))
    result = subprocess.run([str(a) for a in args], cwd=ROOT, text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {args}')
    return result.returncode

if INSTALL_REQUIREMENTS:
    run_command([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'])


## 2. Project Configuration

The configuration file is `configs/config.yaml` in the project root. It is the single source of truth for the data universe and modeling setup. The notebook loaded it in the setup cell, so the displayed horizon, sample ticker, data dates, benchmark, macro sources, and 100-stock universe all come from the same config used by the scripts.

The current setup uses data from `2005-01-01`, which gives enough history for long-horizon labels. The training universe is configured with 100 large/liquid US-listed stocks across sectors. The stock universe is joined with `SPY` benchmark features and macro / alternative proxies for volatility, interest rates, and dollar strength.

The horizon is measured in trading days. The configured horizons are `1`, `5`, `21`, `126`, `252`, `1260`, and `2520`, corresponding approximately to 1 day, 1 week, 1 month, 6 months, 1 year, 5 years, and 10 years.


In [ ]:
source_rows = []
for ticker in config['tickers']:
    source_rows.append({'source_type': 'stock_price', 'name': ticker, 'ticker': ticker})
source_rows.append({'source_type': 'benchmark', 'name': 'market benchmark', 'ticker': config['benchmark_ticker']})
for name, ticker in config.get('macro_tickers', {}).items():
    source_rows.append({'source_type': 'macro_or_alternative', 'name': name, 'ticker': ticker})

print(f'Configuration loaded from: {config_path}')
print(f"Configured stock universe size: {len(config['tickers'])} tickers")
print(f"Configured start date: {config['start_date']}")
print(f"Configured end date: {config['end_date'] or 'latest available'}")
print(f"Available horizons: {available_horizons}")
print(f"Selected notebook horizon: {HORIZON} trading days")
print(f"Dataset cache enabled: {config.get('use_dataset_cache', True)}")
print(f"Force cache rebuild: {config.get('force_rebuild_dataset_cache', False)}")
print(f"Training device setting: {config.get('device', 'auto')}")
print(f"Buy threshold: {float(config['buy_threshold']):.2%}; sell threshold: {float(config['sell_threshold']):.2%}")
display(pd.DataFrame(source_rows))


## 3. Data Collection Design

The raw data is collected by `src/data_download.py` through Yahoo Finance using `yfinance`.

Exact sources used in this project:

- Stock OHLCV history: daily Open, High, Low, Close, Adjusted Close, and Volume for each configured stock ticker.
- Benchmark history: `SPY`, used to add broad market return and volatility context.
- Macro / alternative market data: `^VIX` for market fear / volatility, `^TNX` as a 10-year Treasury yield proxy, and `DX-Y.NYB` for dollar strength.
- Earnings dates: available earnings calendar fields from Yahoo Finance, used only when the provider returns them.

These sources satisfy the requirement to merge historical financial data with non-stock-specific alternative or macro signals. No private API key is required, and the repository does not need to store secrets.


## 4. Data Loading and Processing

The shared pipeline in `src/pipeline.py` builds the supervised learning dataset in this order:

1. Download benchmark data and macro / alternative data for the configured date range.
2. For each stock ticker, download daily OHLCV history.
3. Add technical indicators from price and volume, including returns, moving averages, RSI, MACD, ATR, Bollinger Band features, and realized volatility.
4. Join benchmark features by date.
5. Join macro / alternative features by date and forward-fill known values.
6. Add earnings proximity and latest known earnings surprise fields where available.
7. Create `future_return` and `signal_label` for the selected horizon.
8. Drop rows that are not usable for training because rolling indicators or future labels are missing.
9. Save the processed supervised dataset to `data/cache/full_dataset_h{horizon}.csv` so the next run can reuse it.

Label mining / label creation: the label is mined from the future price movement after the selected horizon. The horizon is in trading days, so horizon 21 means 21 market sessions later, about one month. The notebook compares the adjusted close 21 trading days later to today's adjusted close. If the future return is at least `buy_threshold` then `signal_label = 2` meaning BUY. If the future return is at most `sell_threshold` then `signal_label = 0` meaning SELL. Otherwise `signal_label = 1` meaning HOLD. This creates supervised labels from historical data without manually labeling examples.

What is each row? Each row is one stock on one trading date. It contains that stock's price/volume data for the date, engineered technical indicators known by that date, joined benchmark and macro values for the same date, earnings context, and the future label used for training. The row is the model's learning example: features are the inputs, while `future_return` and `signal_label` are the targets.


In [ ]:
from src.baselines import add_sma_crossover_baseline
from src.features import get_feature_columns
from src.pipeline import build_or_load_dataset_for_tickers

processed_dir = ROOT / 'data' / 'processed'
cache_dir = ROOT / 'data' / 'cache'
processed_dir.mkdir(parents=True, exist_ok=True)
cache_dir.mkdir(parents=True, exist_ok=True)

dataset_path = processed_dir / f'full_dataset_h{HORIZON}.csv'
dataset_cache_path = cache_dir / f'full_dataset_h{HORIZON}.csv'

full_df, feature_columns = build_or_load_dataset_for_tickers(
    tickers=config['tickers'],
    benchmark_ticker=config['benchmark_ticker'],
    start_date=config['start_date'],
    end_date=config['end_date'],
    prediction_horizon=HORIZON,
    buy_threshold=float(config['buy_threshold']),
    sell_threshold=float(config['sell_threshold']),
    macro_tickers=config.get('macro_tickers'),
    cache_path=dataset_cache_path,
    use_cache=bool(config.get('use_dataset_cache', True)),
    force_rebuild=bool(config.get('force_rebuild_dataset_cache', False)) or REBUILD_DATASET,
)

full_df = add_sma_crossover_baseline(full_df)
full_df.to_csv(dataset_path)
df = full_df.reset_index().rename(columns={'index': 'Date'})

if 'Date' not in df.columns:
    df = df.rename(columns={df.columns[0]: 'Date'})
df['Date'] = pd.to_datetime(df['Date'])

print(f'Dataset cache: {dataset_cache_path}')
print(f'Processed dataset copy: {dataset_path}')
print(f'Rows: {len(df):,}')
print(f'Columns: {len(df.columns):,}')
print(f'Feature columns: {len(feature_columns):,}')
display(df.head())


## 5. Data Validation Test

Before training, the notebook checks the first project requirement directly: the processed dataset must contain at least three years of usable history and must include merged external macro / alternative features. It also checks that the supervised labels exist and that selected model features are not missing after processing.


In [ ]:
date_coverage_years = (df['Date'].max() - df['Date'].min()).days / 365.25
macro_columns = [column for column in df.columns if column.startswith('macro_')]
benchmark_columns = [column for column in df.columns if column.startswith('benchmark_')]
label_values = sorted(df['signal_label'].dropna().astype(int).unique().tolist())
feature_missing_rate = df[feature_columns].isna().mean().sort_values(ascending=False).head(10)

validation_summary = pd.DataFrame(
    [
        {'check': 'usable rows', 'value': len(df), 'passes': len(df) > 0},
        {'check': 'start date', 'value': str(df['Date'].min().date()), 'passes': True},
        {'check': 'end date', 'value': str(df['Date'].max().date()), 'passes': True},
        {'check': 'coverage years', 'value': round(date_coverage_years, 2), 'passes': date_coverage_years >= 3.0},
        {'check': 'stock tickers', 'value': df['Ticker'].nunique(), 'passes': df['Ticker'].nunique() >= 1},
        {'check': 'macro / alternative columns', 'value': len(macro_columns), 'passes': len(macro_columns) >= 3},
        {'check': 'benchmark columns', 'value': len(benchmark_columns), 'passes': len(benchmark_columns) >= 1},
        {'check': 'labels present', 'value': label_values, 'passes': set(label_values).issubset({0, 1, 2}) and len(label_values) > 0},
        {'check': 'feature missing values', 'value': float(df[feature_columns].isna().sum().sum()), 'passes': df[feature_columns].isna().sum().sum() == 0},
    ]
)

display(validation_summary)
display(feature_missing_rate.rename('missing_rate').reset_index().rename(columns={'index': 'feature'}))

assert date_coverage_years >= 3.0, 'Dataset must contain at least 3 years of usable rows.'
assert len(macro_columns) >= 3, 'Dataset must include macro / alternative features.'
assert len(benchmark_columns) >= 1, 'Dataset must include benchmark features.'
assert df[feature_columns].isna().sum().sum() == 0, 'Feature matrix contains missing values.'
assert set(label_values).issubset({0, 1, 2}) and len(label_values) > 0, 'Labels must be BUY/HOLD/SELL ids.'


## 6. Source Code Validation

These checks confirm that the scripts and `src` package compile. The unit test validates the data-processing contract with synthetic data, so it can run without live downloads.


In [ ]:
run_command([sys.executable, '-m', 'py_compile', 'train.py', 'train_xgboost.py', 'compare_results.py', 'compare_models.py', 'evaluate_saved_model.py', 'predict.py'])
run_command([sys.executable, '-m', 'compileall', 'src'])
run_command([sys.executable, '-m', 'unittest', 'tests.test_data_pipeline_validation'])


## Feature Engineering Summary

The shared pipeline builds technical indicators, benchmark features, macro / alternative features, earnings features, and future-return labels. The split used later is chronological to avoid training on future data.


## LSTM Training

The LSTM is the sequential model. It uses rolling windows and has two heads: one for BUY / HOLD / SELL classification and one for future return regression.

In [ ]:
if RUN_TRAINING:
    run_command([sys.executable, 'train.py', '--horizon', HORIZON])
else:
    print('Skipping LSTM training because RUN_TRAINING = False')

## XGBoost Training

XGBoost is the second model family. It uses the same dataset pipeline and chronological split, then saves a classifier, regressor, metrics, predictions, and backtest results.

In [ ]:
if RUN_TRAINING:
    run_command([sys.executable, 'train_xgboost.py', '--horizon', HORIZON])
else:
    print('Skipping XGBoost training because RUN_TRAINING = False')

## Model Comparison

This creates the comparison CSV and Markdown report under `reports/`.

In [ ]:
run_command([sys.executable, 'compare_results.py', '--horizon', HORIZON])

In [ ]:
comparison_path = ROOT / 'reports' / f'model_comparison_h{HORIZON}.csv'
comparison = pd.read_csv(comparison_path)
comparison

## Evaluation Summaries

Print the saved statistical metrics, backtest metrics, and signal distribution for each model.

In [ ]:
run_command([sys.executable, 'evaluate_saved_model.py', '--model', 'lstm', '--horizon', HORIZON])
run_command([sys.executable, 'evaluate_saved_model.py', '--model', 'xgboost', '--horizon', HORIZON])

## EDA

After training, the processed dataset can be inspected directly. This gives a quick view of rows, date coverage, tickers, label balance, and return distribution.

In [ ]:
dataset_path = ROOT / 'data' / 'processed' / f'full_dataset_h{HORIZON}.csv'
df = pd.read_csv(dataset_path, parse_dates=['Date'] if 'Date' in pd.read_csv(dataset_path, nrows=0).columns else None)

date_col = 'Date' if 'Date' in df.columns else df.columns[0]
df[date_col] = pd.to_datetime(df[date_col])

summary = {
    'rows': len(df),
    'columns': len(df.columns),
    'start_date': str(df[date_col].min().date()),
    'end_date': str(df[date_col].max().date()),
    'tickers': ', '.join(sorted(df['Ticker'].unique())) if 'Ticker' in df.columns else 'n/a',
}
pd.DataFrame([summary])

In [ ]:
label_names = {0: 'SELL', 1: 'HOLD', 2: 'BUY'}
label_counts = df['signal_label'].map(label_names).value_counts().rename_axis('label').reset_index(name='count')
label_counts['share'] = label_counts['count'] / label_counts['count'].sum()
label_counts

In [ ]:
df[['future_return']].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])

## Backtest Comparison

The backtest treats each prediction as an academic simulated trade over the selected horizon, subtracts transaction cost and slippage, and compares performance against buy-and-hold.

In [ ]:
lstm_backtest = pd.read_csv(ROOT / 'reports' / f'backtest_results_h{HORIZON}.csv')
xgb_backtest = pd.read_csv(ROOT / 'reports' / f'backtest_results_xgboost_h{HORIZON}.csv')

display(lstm_backtest.tail())
display(xgb_backtest.tail())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(pd.to_datetime(lstm_backtest['date']), lstm_backtest['equity'], label='LSTM strategy')
plt.plot(pd.to_datetime(xgb_backtest['date']), xgb_backtest['equity'], label='XGBoost strategy')
plt.plot(pd.to_datetime(xgb_backtest['date']), xgb_backtest['buy_and_hold_equity'], label='Buy and hold', linestyle='--')
plt.title(f'Backtest Equity Comparison - Horizon {HORIZON}')
plt.xlabel('Date')
plt.ylabel('Equity')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Inference

Run one prediction using the trained LSTM artifacts and the same inference feature pipeline.

In [ ]:
run_command([sys.executable, 'predict.py', '--ticker', SAMPLE_TICKER, '--horizon', HORIZON])
pd.read_csv(ROOT / 'reports' / 'latest_predictions.csv')

## Output Files

The main demo-ready files are listed below.

In [ ]:
outputs = [
    ROOT / 'models' / f'stock_advanced_model_h{HORIZON}.pt',
    ROOT / 'models' / f'xgboost_classifier_h{HORIZON}.joblib',
    ROOT / 'models' / f'xgboost_regressor_h{HORIZON}.joblib',
    ROOT / 'reports' / f'metrics_h{HORIZON}.json',
    ROOT / 'reports' / f'metrics_xgboost_h{HORIZON}.json',
    ROOT / 'reports' / f'model_comparison_h{HORIZON}.csv',
    ROOT / 'reports' / f'model_comparison_h{HORIZON}.md',
    ROOT / 'reports' / f'backtest_results_h{HORIZON}.csv',
    ROOT / 'reports' / f'backtest_results_xgboost_h{HORIZON}.csv',
]

pd.DataFrame({'file': [str(path.relative_to(ROOT)) for path in outputs], 'exists': [path.exists() for path in outputs]})

## Risks and Future Work

- Yahoo Finance data can be delayed, revised, incomplete, or unavailable for some tickers.
- The backtest is simplified and does not model market impact, order-book dynamics, liquidity, or production risk controls.
- Results can be unstable across horizons and market regimes.
- Future work can add walk-forward validation, stronger risk management, richer alternative data, and additional model families.

Academic disclaimer: this notebook is for research and simulation only. It is not investment advice.